In [1]:
import io
import sys
import tarfile
from pathlib import Path

import pandas as pd

repo_root = Path.cwd().resolve().parents[1] if Path.cwd().name == "docs" else (Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from mir.common.parser import AIRRParser, VDJdbSlimParser, VDJtoolsParser
from mir.common.repertoire import LocusRepertoire
from mir.utils.notebook_assets import (
    ensure_airr_benchmark,
    find_airr_benchmark_sra_meta,
    find_airr_benchmark_vdjdb_slim,
)

benchmark_root = ensure_airr_benchmark(repo_root, allow_patterns=["sra/**", "vdjdb/**", "vdjtools_lite/**"])
vdjdb_path = find_airr_benchmark_vdjdb_slim(benchmark_root)
sra_tarball, sra_meta_path = find_airr_benchmark_sra_meta(benchmark_root)
vdjtools_candidates = sorted((benchmark_root / "vdjtools_lite").glob("*.txt.gz"))
if not vdjtools_candidates:
    raise FileNotFoundError(f"No VDJtools files found under {benchmark_root / 'vdjtools_lite'}")
vdjtools_path = vdjtools_candidates[0]

print(f"vdjdb_path = {vdjdb_path}")
print(f"sra_tarball = {sra_tarball}")
print(f"sra_meta_path = {sra_meta_path}")
print(f"vdjtools_path = {vdjtools_path}")

/Users/mikesh/vcs/mirpy/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 55 files: 100%|██████████| 55/55 [00:00<00:00, 2093.74it/s]

vdjdb_path = /Users/mikesh/vcs/mirpy/notebooks/assets/large/airr_benchmark/vdjdb/vdjdb-2025-12-29/vdjdb.slim.txt.gz
sra_tarball = /Users/mikesh/vcs/mirpy/notebooks/assets/large/airr_benchmark/sra/samples.tar.gz
sra_meta_path = /Users/mikesh/vcs/mirpy/notebooks/assets/large/airr_benchmark/sra/meta.tsv
vdjtools_path = /Users/mikesh/vcs/mirpy/notebooks/assets/large/airr_benchmark/vdjtools_lite/10months.txt.gz


# Parsing AIRR, VDJtools, and VDJdb Formats

In [2]:
sample = VDJdbSlimParser().parse_file(vdjdb_path, species="HomoSapiens")
rep = sample["TRA"] if "TRA" in sample.loci else next(iter(sample.loci.values()))
print([(c.v_gene, c.junction_aa) for c in rep[:5]])
print(rep)

[('TRAV29/DV5*01', 'AQGLLTGGGNKLTF'), ('TRAV12-2*01', 'ASMFSGGGNEKFTF'), ('TRAV26-2*01', 'ASMYKGGGNEKFTF'), ('TRAV29/DV5*01', 'CAAAAGNMLTF'), ('TRAV13-1*01', 'CAAAAGNTGKLIF')]
LocusRepertoire(locus='TRA', clonotypes=43965, duplicate_count=0)


## Parse AIRR from the SRA benchmark bundle

The benchmark SRA bundle stores mixed-locus AIRR data (TRB, TRA, IGK …) without an
explicit `locus` column.  This cell reads one sample, infers `locus` from the `v_call`
prefix, filters to TRB, then parses with `AIRRParser`.

In [5]:
with tarfile.open(sra_tarball, "r:gz") as tar:
    members = sorted(name for name in tar.getnames() if name.endswith(".tsv"))
    if not members:
        raise FileNotFoundError(f"No AIRR TSV members found in {sra_tarball}")
    target_member = members[0]
    with tar.extractfile(target_member) as raw:
        airr_df = pd.read_csv(io.BytesIO(raw.read()), sep="\t")

# The benchmark SRA bundle stores mixed-locus AIRR data without an explicit
# locus column.  Infer locus from the v_call prefix (e.g. "TRBV…" → "TRB").
_GENE_PREFIX_TO_LOCUS = {"TRBV": "TRB", "TRAV": "TRA", "IGHV": "IGH",
                          "IGKV": "IGK", "IGLV": "IGL", "TRDV": "TRD", "TRGV": "TRG"}
airr_df["locus"] = airr_df["v_call"].str[:4].map(_GENE_PREFIX_TO_LOCUS)
trb_df = airr_df[airr_df["locus"] == "TRB"].copy()

clonotypes = AIRRParser(locus="TRB").parse_inner(trb_df)
rep = LocusRepertoire(clonotypes=clonotypes, locus="TRB")
print(target_member)
print([(c.v_gene, c.junction_aa) for c in rep[:5]])
print(rep)

./SRR10611239.tsv
[('TRBV27*01', 'CASSRSANSPLHF'), ('TRBV29-1*01', 'CSVLGQPTYGYTF'), ('TRBV12-3*01', 'CASERWSSGNTIYF'), ('TRBV29-1*01', 'CSVVKGQGPNQPQHF'), ('TRBV19*01', 'CASSIFQGSYEQYF')]
LocusRepertoire(locus='TRB', clonotypes=267, duplicate_count=312)


Load VDJtools format data from `airr_benchmark/vdjtools_lite`

In [6]:
parser = VDJtoolsParser()
rep = LocusRepertoire(
    clonotypes=parser.parse(str(vdjtools_path)),
    locus="TRB",
)
print([(c.duplicate_count, c.junction_aa) for c in rep[:5]])
print(rep)

[(812, 'CSVEIWDSSYNEQFF'), (704, 'CASSLAPGATNEKLFF'), (571, 'CASSLGENIQYF'), (408, 'CSARTTYGTDIISQHF'), (282, 'CSVGTSEAYEQYF')]
LocusRepertoire(locus='TRB', clonotypes=6525, duplicate_count=14903)
